In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [ ]:
MODEL_DIR = "/kaggle/input/roberta-span"    # folder containing config.json + model + tokenizer files
TEST_CSV  = "/kaggle/input/tweet-sentiment-extraction/test.csv"  # adjust if your test.csv lives elsewhere
SUBMISSION = "submission.csv"
INF_BS = 32
MAX_LEN = 96

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_DIR).to(device).eval()

In [ ]:
test_df = pd.read_csv(TEST_CSV).fillna("")
required = {"textID", "text", "sentiment"}
missing = required - set(test_df.columns)
if missing:
    raise ValueError(f"test.csv must have columns {required}, missing: {missing}")

texts = test_df["text"].astype(str).tolist()
sents = test_df["sentiment"].astype(str).str.lower().str.strip().tolist()

In [ ]:
@torch.no_grad()
def predict_batch(sentiments, texts, tokenizer, model, max_len=MAX_LEN, batch_size=INF_BS):
    preds = []
    N = len(texts)
    for i in range(0, N, batch_size):
        s_batch = [str(s).lower().strip() for s in sentiments[i:i+batch_size]]
        t_batch = [str(t) for t in texts[i:i+batch_size]]

        enc = tokenizer(
            s_batch, t_batch,
            max_length=max_len,
            padding=True,
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        offsets = enc["offset_mapping"]  # (B, L, 2) CPU tensor

        out = model(input_ids=input_ids, attention_mask=attention_mask)
        start_logits = out.start_logits.cpu()  # (B, L)
        end_logits   = out.end_logits.cpu()    # (B, L)

        for b in range(start_logits.size(0)):
            sl = start_logits[b]
            el = end_logits[b]
            offs = offsets[b].tolist()
            text = t_batch[b]

            # valid token indices: skip specials with (0,0)
            valid = [k for k, (a, b_) in enumerate(offs) if b_ > 0]
            if not valid:
                preds.append("")
                continue

            # best span by maximizing sl[i] + el[j], with i <= j (O(L^2), fine for L~96)
            best_score, bi, bj = -1e9, valid[0], valid[0]
            for i2 in valid:
                si = sl[i2].item()
                for j2 in valid:
                    if j2 < i2:
                        continue
                    score = si + el[j2].item()
                    if score > best_score:
                        best_score, bi, bj = score, i2, j2

            s_char, _ = offs[bi]
            _, e_char = offs[bj]
            if e_char < s_char:
                e_char = s_char
            preds.append(text[s_char:e_char])

    return preds

In [ ]:
preds = predict_batch(sents, texts, tokenizer, model)

In [ ]:
assert len(preds) == len(test_df), f"Preds len {len(preds)} != test rows {len(test_df)}"
submission = pd.DataFrame({"textID": test_df["textID"], "selected_text": preds})
submission.to_csv(SUBMISSION, index=False)